### Import Package

In [15]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from tensorflow.keras.callbacks import ModelCheckpoint
from transformers import AutoTokenizer
from transformers import create_optimizer
from transformers import TFAutoModelForSequenceClassification
from transformers import DataCollatorWithPadding

### Read and Split Dataset

In [16]:
training_file = "dataset/train.csv"
test_file = "dataset/test.csv"
output_dir = './model2_outputs'
batch_size = 16

In [17]:
df = load_dataset('csv', data_files = [training_file])
df = df['train'].train_test_split(test_size = 0.1)
df['valid'] = df['test']
df['test'] = load_dataset('csv', data_files = [test_file])['train']

Using custom data configuration default-dacd6034bd90bc38
Reusing dataset csv (/home/feng/.cache/huggingface/datasets/csv/default-dacd6034bd90bc38/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519)
100%|██████████| 1/1 [00:00<00:00, 255.19it/s]
Using custom data configuration default-60f927a822212e41
Reusing dataset csv (/home/feng/.cache/huggingface/datasets/csv/default-60f927a822212e41/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519)
100%|██████████| 1/1 [00:00<00:00, 267.46it/s]


In [18]:
pd.DataFrame(df['train'])

,id,keyword,location,text,target
0,3390,demolition,Arthas US,Doing Giveaway Music Kit Dren Death's Head Dem...,0
1,7932,rainstorm,"Memphis, TN",If you can't have the roar of the waves a rain...,0
2,2429,collide,EspÌ_rito Santo,Maybe if the stars align maybe if our worlds c...,0
3,2387,collapsed,I'm standing behind you,@rokiieee_ the game has officially collapsed,0
4,3119,debris,Campo Grande-MS,[Reuters] Debris confirmed from MH370; relativ...,1
...,...,...,...,...,...
6846,9741,tragedy,Orlando,Back home they mad cause I chill with the whit...,0
6847,8292,rubble,"Columbus, Georgia",'Refuse to let my life be reduced to rubble. W...,0
6848,4788,evacuated,"Portland, Oregon",Evacuation orders lifted for Roosevelt in High...,1
6849,3265,demolish,None,Demolish-deep space etoffe charmeuse clothesle...,0


In [19]:
pd.DataFrame(df['valid'])

,id,keyword,location,text,target
0,290,ambulance,None,What's the police or ambulance number in Lesot...,0
1,7138,military,None,I remember when I worked at Mcdonalds I use to...,0
2,1721,buildings%20burning,None,@SonofLiberty357 all illuminated by the bright...,0
3,8410,sandstorm,USA,Watch This Airport Get Swallowed Up By A Sands...,1
4,5661,floods,South of D.C.,Slip Sliding Away - Flash Floods Info for Writ...,1
...,...,...,...,...,...
757,8942,storm,None,FINALLY a storm,0
758,916,bioterrorism,None,To fight bioterrorism sir.,0
759,2227,chemical%20emergency,None,Google Alert: Emergency units simulate a chemi...,0
760,8860,smoke,None,I wanna drink a little smoke a little,0


In [20]:
pd.DataFrame(df['test'])

,id,keyword,location,text
0,0,None,None,Just happened a terrible car crash
1,2,None,None,"Heard about #earthquake is different cities, s..."
2,3,None,None,"there is a forest fire at spot pond, geese are..."
3,9,None,None,Apocalypse lighting. #Spokane #wildfires
4,11,None,None,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,None,None,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,None,None,Storm in RI worse than last hurricane. My city...
3260,10868,None,None,Green Line derailment in Chicago http://t.co/U...
3261,10874,None,None,MEG issues Hazardous Weather Outlook (HWO) htt...


Padding adds a special padding token to ensure shorter sequences will have the same length as either the longest sequence in a batch or the maximum length accepted by the model.

Truncation works in the other direction by truncating long sequences.

In [21]:
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Stop Warnings
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Use Predefined Distilbert Tokenizer

def tokenize(row):
    return tokenizer(row["text"], padding = "max_length", truncation = True) # Padding and Truncation to Max Model Input Length

origin_columns = set(df["train"].features) # Not yet tokenized dataset
encoded_df = df.map(tokenize, batched = True) # Open Batch Processing, Default Batch Size is 1000
tokenizer_columns = list(set(encoded_df["train"].features) - origin_columns)
print("Columns added by tokenizer:", tokenizer_columns)

100%|██████████| 1/1 [00:00<00:00,  9.62ba/s]

Columns added by tokenizer: ['input_ids', 'attention_mask']


In [22]:
df

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 6851
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text'],
        num_rows: 3263
    })
    valid: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 762
    })
})

In [23]:
encoded_df

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target', 'input_ids', 'attention_mask'],
        num_rows: 6851
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'input_ids', 'attention_mask'],
        num_rows: 3263
    })
    valid: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target', 'input_ids', 'attention_mask'],
        num_rows: 762
    })
})

In [24]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer, return_tensors = "tf")

train_df = encoded_df['train'].to_tf_dataset(
    columns = tokenizer_columns,
    label_cols = ["target"],
    shuffle = True,
    collate_fn = data_collator,
    batch_size = batch_size,
)

val_df = encoded_df['valid'].to_tf_dataset(
    columns = tokenizer_columns,
    label_cols = ["target"],
    shuffle = False,
    batch_size = batch_size,
    collate_fn = data_collator,
)

test_df = encoded_df['test'].to_tf_dataset(
    columns = tokenizer_columns,
    shuffle = False,
    batch_size = batch_size,
    collate_fn = data_collator,
)

test_df

<PrefetchDataset element_spec={'input_ids': TensorSpec(shape=(None, None), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, None), dtype=tf.int64, name=None)}>

In [25]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels = 2
)

model.summary()

Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['activation_13', 'vocab_transform', 'vocab_layer_norm', 'vocab_projector']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier', 'dropout_39', 'pre_classifier']
You should probably TRAIN this model on a down-stream task to be able to use i

Model: "tf_distil_bert_for_sequence_classification_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMai  multiple                 66362880  
 nLayer)                                                         
                                                                 
 pre_classifier (Dense)      multiple                  590592    
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
 dropout_39 (Dropout)        multiple                  0         
                                                                 
Total params: 66,955,010
Trainable params: 66,955,010
Non-trainable params: 0
_________________________________________________________________


In [26]:
num_epochs = 3
batches_per_epoch = len(encoded_df["train"]) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)

optimizer, schedule = create_optimizer(
    init_lr = 2e-5, num_warmup_steps = 0, num_train_steps = total_train_steps
)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True)
model.compile(optimizer = optimizer, loss = loss, metrics = ['accuracy'])

In [27]:
if not os.path.exists(output_dir): # If the file directory doesn't already exists,
    os.makedirs(output_dir) # Make it again

checkpoint_callback = ModelCheckpoint(filepath = output_dir + '/weights.{epoch:02d}.hdf5', monitor = 'val_loss', save_best_only = True, save_weights_only = True)

In [28]:
model.fit(
    train_df,
    validation_data = val_df,
    epochs = 3,
    callbacks = [checkpoint_callback],
)

Epoch 1/3
388/428 [==========================>...] - ETA: 4:07 - loss: 0.4466 - accuracy: 0.8014

### Predict and Output Test Dataset

In [ ]:
test_pred = model.predict(test_df)

In [ ]:
submission = pd.read_csv('dataset/sample_submission.csv')
submission['target'] = np.argmax(test_pred.logits, axis = 1)
submission.to_csv('submission.csv', index = False)